**Consultas a bases de datos SQL con LLMs**

Este programa permite interpretar consultas en lenguaje natural en bases de datos del tipo SQL a través de modelos como GPT. Para esto se siguen los siguientes pasos:

1. Cargar una tabla con datos de venta ("sales.csv").
2. Convertir la tabla a SQL
3. Crear un prompt para realizar una consulta a GPT en la base de datos SQL
4. Retornar la respuesta

Primero, instalamos algunos paquetes para acceder a los modelos:

In [ ]:
!pip install openai

Importamos algunas librerías de propósito general y otras para manejar bases de datos SQL (*sqlalchemy*):

In [24]:
import pandas as pd
import numpy as np
import pprint
from openai import OpenAI
from sqlalchemy import create_engine
from sqlalchemy import text

In [25]:
# Open AI API-key
from google.colab import files
from IPython.display import clear_output

files.upload() # subir archivo con apikey de openai propio
clear_output() # no muestra contenido del apikey

In [26]:
def get_api_key():
    with open('idsa_openai_key.txt', 'r') as fp: #acá reemplazar x el nombre de tu archivo
        key = fp.read()
    return key

# Enter your OpenAI API key here
openai_api_key = get_api_key()

In [27]:
!wget https://github.com/palasatenea66/DATASETS/raw/main/sales.csv

--2025-05-23 00:29:48--  https://github.com/palasatenea66/DATASETS/raw/main/sales.csv
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/palasatenea66/DATASETS/main/sales.csv [following]
--2025-05-23 00:29:48--  https://raw.githubusercontent.com/palasatenea66/DATASETS/main/sales.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 527958 (516K) [application/octet-stream]
Saving to: ‘sales.csv.1’

sales.csv.1         100%[===================>] 515.58K  --.-KB/s    in 0.007s  

2025-05-23 00:29:48 (76.5 MB/s) - ‘sales.csv.1’ saved [527958/527958]



Definimos una función para inicializar la base de datos SQL con los datos de la tabla de ventas "sales.csv":

In [28]:
def InicializarBD():
   temp = create_engine("sqlite:///:memory:", echo=True)
   data =df.to_sql(name="Sales", con=temp)
   return(data, temp)

Definimos la función **CrearPrompt(df)** que le informa a GPT sobre los datos **df** y sus propiedades que utilizaremos (se crea una tabla con todas las columnas de los datos iniciales):

In [9]:
def CrearPrompt(df):
  prompt = '''### sqlite SQL table:
#
# Sales({})
#
'''.format(",".join(str(x) for x in df.columns))
  return(prompt)

Creamos una función CombinarPrompt(df,consulta) para combinar la consulta del usuario **consulta** con la estructura de la tabla **df** con el string adicional **"Una consulta a responder: ”** seguido por la palabra clave “**Select**” de modo que GPT entienda la consulta correctamente:

In [29]:
def CombinarPrompt(df,consultaPrompt):
   defi = CrearPrompt(df)
   query_string = f'### Una consulta a responder: {consultaPrompt}\nSELECT'
   return(defi+query_string)

Definimos la función **GenerarRespuestaGPT(df,nlp_text)**  para invocar a la API de OpenAI y utilizar el modelo *"gpt-3.5-turbo-instruct"* para entregar los resultados, y algunos otros parámetros tales como la temperatura y número máximo de tokens a retornar:

In [30]:
def GenerarRespuestaGPT(df,nlp_text):
    client=OpenAI(api_key=openai_api_key)
    response = client.completions.create(
      model="gpt-3.5-turbo-instruct",
      prompt=CombinarPrompt(df,nlp_text),
      max_tokens=150,
      n=1,
      stop=['#',';'],
      temperature=0.7,
    )
    return(response)

Creamos la función **ManejarRespuesta(respuesta)** que
analiza la respuesta y se la pasa a la base de datos:

In [31]:
def ManejarRespuesta(respuesta):
   query = response.choices[0].text
   if query.startswith(" "):
       query = "Select"+query
   return(query)

Ahora, comenzamos el programa principal:

In [32]:
# Cargar datos de ventas "sales.csv"
df = pd.read_csv('/content/sales.csv', encoding="latin1")
data, temp = InicializarBD()

2025-05-23 00:34:03,330 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2025-05-23 00:34:03,337 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("Sales")


INFO:sqlalchemy.engine.Engine:PRAGMA main.table_info("Sales")


2025-05-23 00:34:03,339 INFO sqlalchemy.engine.Engine [raw sql] ()


...

INFO:sqlalchemy.engine.Engine:SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite~_%' ESCAPE '~' ORDER BY name


2025-05-23 00:34:03,441 INFO sqlalchemy.engine.Engine [raw sql] ()


INFO:sqlalchemy.engine.Engine:[raw sql] ()


2025-05-23 00:34:03,444 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


In [37]:
nlp_text = input("Ingrese consulta para GPT:")
response = GenerarRespuestaGPT(df,nlp_text)

Ingrese consulta para GPT:quiero saber cuantos aviones se vendieron cada año


In [38]:
print(response.choices[0].text)

 
    YEAR_ID,
    SUM(QUANTITYORDERED) AS planes_sold
FROM
    Sales
WHERE
    PRODUCTLINE = 'Planes'
GROUP BY YEAR_ID


In [39]:
# Ahora pasamos la respuesta SQL a la base de datos para generar el resultado
with temp.connect() as conn:
  result = conn.execute(text(ManejarRespuesta(response)))

2025-05-23 00:39:34,632 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2025-05-23 00:39:34,635 INFO sqlalchemy.engine.Engine Select 
    YEAR_ID,
    SUM(QUANTITYORDERED) AS planes_sold
FROM
    Sales
WHERE
    PRODUCTLINE = 'Planes'
GROUP BY YEAR_ID


INFO:sqlalchemy.engine.Engine:Select 
    YEAR_ID,
    SUM(QUANTITYORDERED) AS planes_sold
FROM
    Sales
WHERE
    PRODUCTLINE = 'Planes'
GROUP BY YEAR_ID


2025-05-23 00:39:34,636 INFO sqlalchemy.engine.Engine [generated in 0.00410s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00410s] ()


2025-05-23 00:39:34,639 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK


In [40]:
# Mostramos los resultados de la consulta SQL
result.all()

[(2003, 2996), (2004, 5524), (2005, 2207)]